# Data Cleaning 

This notebook cleans the raw data available in data/raw and writes the clean version back to the folder data/processed. 

In [14]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from c08_farming_exit import config, features, data_cleaning, mappings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

In [15]:
COUNTRIES = {
    "Botswana": config.RAW_DATA_DIR / "Botswana",
    "Kenya":    config.RAW_DATA_DIR / "Kenya",
    "Namibia":  config.RAW_DATA_DIR / "Namibia",
    "Tanzania": config.RAW_DATA_DIR / "Tanzania",
    "Zambia":   config.RAW_DATA_DIR / "Zambia",
}

## 1. Database

In [16]:
#THE DATABASE CONSISTS OF ADULTS ONLY
database = []

for country, base_path in COUNTRIES.items():
    identifying_info    = data_cleaning.load_and_preprocess(base_path, f"{country}_identifying_info.csv",                  features.IDENTIFYING_INFO_2023)
    hh_members          = data_cleaning.load_and_preprocess(base_path, f"{country}_household_members_characterstics.csv",  features.HH_MEMBERS_2023)
    hh_members          = data_cleaning.create_education_features(hh_members, country)

    merge = identifying_info.merge(hh_members, on=["interview_key"], how="inner")
    
    filtered = merge[
        merge["relation_to_head"].isin([    "Self/Head", 
                                            "Wife/Husband", 
                                            "Son/Daughter-In-Law", 
                                            "Sister/Brother", 
                                            "Mother/Father", 
                                            "Brother/Sister-In-Law", 
                                            "Grandfather/Mother", 
                                            "Father/Mother-In-Law"]) &
                                        (merge["age"] >= 18)
    ]

    database.append(filtered)

df_database = pd.concat(database, ignore_index=True)

#CREATE PERSONAL IDENTIFIER 
df_database["personal_id"] = df_database["country"] + "_" + df_database["interview_key"].astype(str) + "_" + df_database["members_id"].astype(str)

#FINAL SORTING
df_database = df_database[['country', 'region', 'district', 'enumeration_area', 'personal_id', 'interview_key', 'relation_to_head', 'gender', 'age', 'years_of_schooling', 'education_level']] \
                .query("country != 'YES+A112:L126+A112:C126'")

[Botswana_identifying_info.csv] Missing columns: ['ea', 'region']
mapping: Botswana - Mapping dictionary of 'education_level' is incomplete: Dropped 216 rows with NaN in 'years_of_schooling'.
mapping: Botswana - Mapping dictionary of 'education_level' is incomplete: Dropped 216 rows with NaN in 'education_level'.
mapping: Kenya - Mapping dictionary of 'education_level' is incomplete: Dropped 506 rows with NaN in 'years_of_schooling'.
mapping: Kenya - Mapping dictionary of 'education_level' is incomplete: Dropped 506 rows with NaN in 'education_level'.
[Namibia_identifying_info.csv] Missing columns: ['dist']
mapping: Namibia - Mapping dictionary of 'education_level' is incomplete: Dropped 441 rows with NaN in 'years_of_schooling'.
mapping: Namibia - Mapping dictionary of 'education_level' is incomplete: Dropped 441 rows with NaN in 'education_level'.
mapping: Tanzania - Mapping dictionary of 'education_level' is incomplete: Dropped 385 rows with NaN in 'years_of_schooling'.
mapping: Tan

## 2. Features

### 2.1 Creating HH-Level Features

In [ ]:
hh_features = []

for country, base_path in COUNTRIES.items():
    #ONE OBSERVATION PER HH
    land_ownership          = data_cleaning.load_and_preprocess(base_path, f"{country}_land_ownership_and_access.csv",         features.LAND_OWNERSHIP_ACCESS_2023)
    land_ownership          = data_cleaning.convert_land_sizes_to_acres(land_ownership, country, "land_measurement", mappings.acres_conversion_factors)
    crop_expenditure        = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_on_crops.csv",              features.CROP_EXPENDITURE_2023)
    lifestock_grazing       = data_cleaning.load_and_preprocess(base_path, f"{country}_grazing_patterns_and_schemes.csv",      features.LIFESTOCK_GRAZING_2023)
    livestock_income        = data_cleaning.load_and_preprocess(base_path, f"{country}_income_livestock.csv",                  features.LIFESTOCK_INCOME_2023)
    livestock_expenditure   = data_cleaning.load_and_preprocess(base_path, f"{country}_expenditure_livestock.csv",             features.LIFESTOCK_EXPENDITURE_2023)
    housing_conditions      = data_cleaning.load_and_preprocess(base_path, f"{country}_housing_conditions.csv",                features.HOUSING_CONDITIONS_2023)
    energy_access           = data_cleaning.load_and_preprocess(base_path, f"{country}_access_to_energy.csv",                  features.ENERGY_ACCESS_2023)
    internet_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_internet_access.csv",                   features.INTERNET_ACCESS_2023)
    social_network          = data_cleaning.load_and_preprocess(base_path, f"{country}_other_household_social_network.csv",    features.SOCIAL_NETWORK_2023)
    social_embeddedness     = data_cleaning.load_and_preprocess(base_path, f"{country}_social_embeddedness.csv",               features.SOCIAL_EMBEDDEDNESS_2023)
    food_insecurity         = data_cleaning.load_and_preprocess(base_path, f"{country}_food_insecurity_experiance_scale.csv",  features.FOOD_INSECURITY_2023)
    road_connectivity       = data_cleaning.load_and_preprocess(base_path, f"{country}_road_connectivity.csv",                 features.ROAD_CONNECTIVITY_2023)

    #MANY OBSERVATIONS PER HH
    if country == "Tanzania":
        #Tanzania's market access data is stored in value_chains.csv.
        market_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_vale_chains.csv",                       features.MARKET_ACCESS_2023)
    else:
        market_access         = data_cleaning.load_and_preprocess(base_path, f"{country}_market_access.csv",                     features.MARKET_ACCESS_2023)
    market_access             = data_cleaning.resolve_duplicates(market_access, key_col="interview_key", sort_col="crop_contract_crop_type", ascending=True)
    crop_production           = data_cleaning.load_and_preprocess(base_path, f"{country}_crop_production.csv",                   features.CROP_PRODUCTION_2023)
    crop_production           = data_cleaning.crop_production_manual_cleaning(crop_production, "interview_key")
    livestock_ownership       = data_cleaning.load_and_preprocess(base_path, f"{country}_livestock_ownership.csv",               features.LIVESTOCK_OWNERSHIP_2023)
    livestock_ownership       = data_cleaning.create_livestock_features(livestock_ownership, country)
    assets_owned              = data_cleaning.load_and_preprocess(base_path, f"{country}_assets.csv",                            features.ASSETS_OWNED_2023)
    assets_owned              = data_cleaning.create_asset_features(assets_owned)
    other_income              = data_cleaning.load_and_preprocess(base_path, f"{country}_other_income.csv",                      features.OTHER_INCOME_SOURCES_2023)
    other_income              = data_cleaning.create_other_income_features(other_income, country)
    shocks_and_coping         = data_cleaning.load_and_preprocess(base_path, f"{country}_shocks_and_coping.csv",                 features.SHOCKS_AND_COPING_2023)
    shocks                    = data_cleaning.create_shock_features(shocks_and_coping, country, "shock_type_affected_last_12_months", mappings.shock_categories, index="interview_key")
    coping                    = data_cleaning.create_coping_features(shocks_and_coping, country, mappings.likelihood)



    #SOME TABLES ARE NOT AVAILABLE FOR EACH COUNTRY: optional_merges solves this as it only merges available tables
    merges = [
        (land_ownership,          ["interview_key"],     "outer"),
        (crop_expenditure,        ["interview_key"],     "outer"),
        (lifestock_grazing,       ["interview_key"],     "outer"),
        (livestock_income,        ["interview_key"],     "outer"),
        (livestock_expenditure,   ["interview_key"],     "outer"),
        (housing_conditions,      ["interview_key"],     "outer"),
        (energy_access,           ["interview_key"],     "outer"),
        (internet_access,         ["interview_key"],     "outer"),
        (social_network,          ["interview_key"],     "outer"),
        (social_embeddedness,     ["interview_key"],     "outer"),
        (food_insecurity,         ["interview_key"],     "outer"),
        (road_connectivity,       ["interview_key"],     "outer"),
        (market_access,           ["interview_key"],     "outer"),
        (crop_production,         ["interview_key"],     "outer"),
        (livestock_ownership,     ["interview_key"],     "outer"),
        (assets_owned,            ["interview_key"],     "outer"),
        (other_income,            ["interview_key"],     "outer"),
        (shocks,                  ["interview_key"],     "outer"),
        (coping,                  ["interview_key"],     "outer"),
    ]

    df_help = None

    for df, keys, how in merges:
        if df_help is None:
            df_help = df
        else:
            df_help = df_help.merge(df, on=keys, how=how)
                
    #Adding the country to the table for identification
    df_help.insert(0, "country", country)

    hh_features.append(df_help)

df_hh_features = pd.concat(hh_features, ignore_index=True)



mapping: Botswana - Mapping dictionary of 'shock_future_likelihood_change_income_source' is incomplete: Dropped 6 rows with NaN in 'shock_future_likelihood_change_income_source'.
convert_land_sizes_to_acres: Tanzania - Dropped 1 rows with NaN in 'land_measurement'
[Tanzania_vale_chains.csv] Missing columns: ['markt_buyer']


In [18]:
missing_by_country = (
    df_hh_features
    .groupby('country')
    .apply(lambda g: g.isna().mean())
    .sort_index()
)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(missing_by_country)


C:\Users\localuser\AppData\Local\Temp\ipykernel_32720\763753577.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.isna().mean())


,country,interview_key,land_measurement,land_cropland_ownership_status,land_residential_ownership_status,land_used_as_collateral,land_number_of_plots,land_size_cropland_acres,land_size_fallow_acres,land_size_agroforestry_forestry_acres,land_size_pasture_acres,land_size_residential_acres,land_size_lodge_camp_acres,crop_exp_seeds_last_12_months,crop_exp_fertilizer_last_12_months,crop_exp_pesticide_last_12_months,crop_exp_machinery_last_12_months,crop_exp_hired_labor_last_12_months,crop_exp_land_rental_last_12_months,crop_exp_transport_last_12_months,crop_exp_other_last_12_months,grazing_distance_in_min,grazing_time_in_min,grazing_time_hours_per_day,grazing_land_ownership_status,grazing_land_number_hh_sharing,grazing_land_permit,grazing_land_permit_price,grazing_land_use_duration_in_years,grazing_land_challenges_little_gras,grazing_land_challenges_prosopis_parthenium,grazing_land_challenges_other_pastoralists,grazing_land_challenges_ethnic_conflict,grazing_land_challenges_tension_conflict,grazing_land_challenges_theft,grazing_land_challenges_raiding,grazing_land_challenges_no_water,grazing_land_challenges_too_far,grazing_land_challenges_expensive,grazing_land_challenges_None,livestock_products_sold_last_12_months,livestock_products_sold_meat,livestock_products_sold_milk,livestock_products_sold_cheese,livestock_products_sold_yogurt,livestock_products_sold_wool,livestock_products_sold_honey_wax,livestock_products_sold_eggs,livestock_products_buyer,livestock_products_market_type,livestock_market_distance_in_km,livestock_income_last_12_months,livestock_contract,livestock_exp_feed_fodder,livestock_exp_rent_gazing_land,livestock_exp_veterinary_services,livestock_exp_shelter,livestock_exp_hired_labor,house_room_number,house_roof_material,house_wall_material,house_floor_material,house_water_source,house_toilet_type,house_energy_source,house_energy_source_for_cooking,house_energy_source_for_lighting,internet_access,internet_access_at_home,mobile_money_access,membership_farmers_group,membership_agricultural_cooperative,my_life_course_depends_on_me,success_is_hard_work,ability_is_more_important_than_effort,my_plans_will_work,I_can_shape_my_future_positively,I_am_optimistic_about_my_future,I_am_optimistic_about_my_familys_future,worry_about_job_loss_or_economic_livelihood,sufficent_food_number_of_month,road_type,road_condition,road_distance_in_minutes,market_output_distance_in_km,market_input_distance_in_km,market_type,crop_contract,crop_contract_crop_type,subsidy,subsidy_type_seeds,subsidy_type_fertilizer,subsidy_type_agro_chemicals,subsidy_type_interest_free_loan,subsidy_supplier_government,subsidy_supplier_ngos,subsidy_supplier_company,market_output_distance_in_km_missing,market_input_distance_in_km_missing,crop_harvested,crop_sale,crop_storage,crop_buyer_market,crop_buyer_trader,crop_buyer_cooperative,crop_buyer_commercial_farm,crop_buyer_hospitality,crop_buyer_government,crop_organic_fertilizer,crop_inorganic_fertilizer,crop_pesticides,crop_tractor,crop_home_consumption,crop_sale_revenue,livestock_number_owned_tlu,livestock_number_lost_disease_theft_tlu,livestock_number_lost_wildlife_attack_tlu,livestock_revenue_sold,asset_value,other_income_amount_yearly,shock_type_affected_last_12_months_crop_failure,shock_type_affected_last_12_months_drought,shock_type_affected_last_12_months_floods,shock_type_affected_last_12_months_illness_death,shock_type_affected_last_12_months_livestock_loss,shock_type_affected_last_12_months_other,shock_type_affected_last_12_months_price_shock,shock_coping_strategy_relatives_friends,shock_coping_strategy_government,shock_coping_strategy_food_reduction,shock_coping_strategy_changed_cropping_practices,shock_coping_strategy_more_employment,shock_coping_strategy_hh_member_migration,shock_coping_strategy_savings,shock_coping_strategy_insurance,shock_coping_strategy_credit,shock_coping_strategy_sold_hh_assets,shock_coping_strategy_sold_livestock,shock_coping_strategy_migration,shock_future_likelihood_change_income_so

In [24]:
df_hh_features['interview_key'].nunique()

3118

In [25]:
len(df_hh_features)

3118

In [19]:
for col in df_hh_features.columns:
    print(col)

country
interview_key
land_measurement
land_cropland_ownership_status
land_residential_ownership_status
land_used_as_collateral
land_number_of_plots
land_size_cropland_acres
land_size_fallow_acres
land_size_agroforestry_forestry_acres
land_size_pasture_acres
land_size_residential_acres
land_size_lodge_camp_acres
crop_exp_seeds_last_12_months
crop_exp_fertilizer_last_12_months
crop_exp_pesticide_last_12_months
crop_exp_machinery_last_12_months
crop_exp_hired_labor_last_12_months
crop_exp_land_rental_last_12_months
crop_exp_transport_last_12_months
crop_exp_other_last_12_months
grazing_distance_in_min
grazing_time_in_min
grazing_time_hours_per_day
grazing_land_ownership_status
grazing_land_number_hh_sharing
grazing_land_permit
grazing_land_permit_price
grazing_land_use_duration_in_years
grazing_land_challenges_little_gras
grazing_land_challenges_prosopis_parthenium
grazing_land_challenges_other_pastoralists
grazing_land_challenges_ethnic_conflict
grazing_land_challenges_tension_conflict


### 2.2 Creating Individual-Level Features

In [14]:
individual_features = []

for country, base_path in COUNTRIES.items():
    #NO CLEANING NECESSARY 

    #CLEANING NECESSARY


    #SOME FEATURES ARE NOT AVAILABLE FOR EACH COUNTRY: optional_merges solves this as it only merges available features
    optional_merges = [
    ]

    df_help = None

    for df, keys, how in optional_merges:
        if df is not None:
            if df_help is None:
                df_help = df
            else:
                df_help = df_help.merge(df, on=keys, how=how)

    individual_features.append(df_help)

df_individual_features = pd.concat(individual_features, ignore_index=True)


ValueError: All objects passed were None

## 3. Write clean data to data/raw folder

In [ ]:
df.to_csv(config.PROCESSED_DATA_DIR / "clean_data.csv", index=False)